In [ ]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots
const bb = 120
const aa = 40
const N = 1_000_000
const I0 = 10
const n_iter = 1_000_000
include("functions.jl")
Random.seed!(2025)

data_file_for(tag::Symbol) =
    tag === :memoryless  ? "data/memoryless.csv" :
    tag === :sliding     ? "data/sliding_kmax14.csv" :        
    tag === :powerlaw    ? "data/powerlaw_lambdaP.csv" :
    tag === :exponential ? "data/exponential_lambdaE.csv" :
    tag === :reciprocal  ? "data/reciprocal_lambdaR.csv" :     
    error("Unknown data tag: $tag")

model_tag_sym = :sliding
data_tag_sym  = :sliding

KMAX_UPPER = 30 

data_path = data_file_for(data_tag_sym)
raw, hdr = DelimitedFiles.readdlm(data_path, ',', header=true)
raw = Matrix{Float64}(raw)
tau = size(raw, 2) ÷ 3

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"]
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]   
else
    error("Unknown model tag: $(model_tag_sym)")
end

const FIXED_KMAX = 14

i = 9
c = 1
Istar_obs = Vector{Int}(round.(Int, raw[i, 1:tau]))

Random.seed!(2025 + i * 100 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,  k_max_fixed=FIXED_KMAX)

    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_sim_$(i)_chain_$(c)_k14.csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods
    loglik_filename = "loglik_sim_$(i)_chain_$(c)_k14.csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))

    el = Dates.value(Dates.now() - t0) / 1000
    @info(@sprintf("Dataset %03d chain %d (k=14): ok in %.2fs", i, c, el))
catch err
    el = Dates.value(Dates.now() - t0) / 1000
    @warn(@sprintf("Dataset %03d chain %d (k=14): ERROR - %s after %.2fs", i, c, err))
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding] iter 1000/1000000 elapsed=3.7s, rate=0.136, medians=[0.764, 0.00434, 0.309], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=8.9s, rate=0.123, medians=[0.775, 0.00433, 0.312], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=12.2s, rate=0.118, medians=[0.781, 0.00432, 0.313], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=15.4s, rate=0.116, medians=[0.783, 0.00431, 0.314], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=18.6s, rate=0.116, medians=[0.785, 0.00431, 0.314], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=21.9s, rate=0.114, medians=[0.786, 0.00431, 0.315], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=25.1s, rate=0.110, medians=[0.787, 0.00432, 0.316], std=[0.0035, 0.000035, 0.0035] [ADAPT]
[ Info: [sliding] iter 8000/1000000 elapsed=28.4s, rate=0